# Reproduction guide

Start with [`hybrid_rag_benchmark.ipynb`](hybrid_rag_benchmark.ipynb) (also available as `source_notebook_sanitized.ipynb` for compatibility). It loads the full packaged `../data/chunks.csv` corpus and `../data/expanded_tickets.csv`, computes a local BM25 baseline, and evaluates the primary target at document level. No API credential or raw PDF is needed for that baseline.

The original 51-cell Colab experiment was audited and replaced: the prior `source_notebook_sanitized.ipynb` had simply retained the 51 original cells with outputs cleared, including failed attempts, duplicate trial-and-error runs, stale Drive paths and credential prompts. The cleaned notebook retains none of those source cells or outputs.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../data") if Path("../data/chunks.csv").exists() else Path("data")
chunks = pd.read_csv(DATA / "chunks.csv")
tickets = pd.read_csv(DATA / "expanded_tickets.csv")
assert len(chunks) == 713
assert len(tickets) == 250
assert tickets["ticket_id"].is_unique
print(f"Validated {len(chunks)} full-text chunks and {len(tickets)} labeled tickets")
print("Chunk columns:", chunks.columns.tolist())
print("Ticket columns:", tickets.columns.tolist())


## Evaluation contract

The main notebook uses `customer_query` as query text, canonicalizes each policy ID as the `source_file` basename without `.pdf`, and compares that to `primary_expected_doc_id`. Rankings deduplicate at policy-document level. Secondary expected documents are available in the data but are not included in the main notebook's primary-target Recall@K/MRR metrics.

The main notebook's BM25 implementation uses the complete chunk `text`. Dense encoders and rerankers are intentionally not executed from this guide: their weights/checkpoints were not supplied. The measured results reported in the README remain historical reported results, not a promise that this included BM25 baseline reproduces all original tables.


## Corpus ingestion module

`../src/phase1_ingest.py` is included for re-ingestion if the original PDFs are separately available. Example: `python src/phase1_ingest.py --input-dir ./your-pdfs --output-dir ./phase1_output`. The packaged `data/chunks.csv` is the complete pre-generated chunk corpus and is sufficient to run the notebook without PDFs.
